## Extract Frames and get FPS

In [1]:
import cv2
import os

def extract_frames(video_path, output_dir, target_size=(384, 288)):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, target_size)
        cv2.imwrite(f"{output_dir}/{frame_idx:06d}.jpg", frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 90])
        frame_idx += 1

    cap.release()
    print(f"Extracted {frame_idx} frames @ {fps} FPS")
    return fps

In [3]:
extract_frames(r"C:\Users\elena.graf\Desktop\AA_Deep_Learning\gate-to-gate-tagging\data\swsk_videos\251025_Gut_Soelden_W_GS_run_1.mp4", r"C:\Users\elena.graf\Desktop\AA_Deep_Learning\gate-to-gate-tagging\data\frames")

Extracted 3144 frames @ 30.0 FPS


30.0

In [4]:
"""
extract_frames.py

Extracts frames from a skiing video at native FPS and saves them as JPEG files.
Resizes to 384x288 as required by the HRNet pose estimator.

Usage:
    python extract_frames.py --video path/to/video.mp4 --out data/frames/run_01
    python extract_frames.py --video path/to/video.mp4 --out data/frames/run_01 --size 640 480
    python extract_frames.py --video path/to/video.mp4 --out data/frames/run_01 --quality 95
"""

import cv2
import os
import json
import argparse
import time


def extract_frames(
    video_path: str,
    output_dir: str,
    target_size: tuple[int, int] = (384, 288),
    jpeg_quality: int = 90,
) -> dict:
    """
    Extract all frames from a video, resize, and save as JPEGs.

    Args:
        video_path:   Path to the input .mp4 (or any OpenCV-readable format).
        output_dir:   Directory where frames will be saved as 000001.jpg, etc.
        target_size:  (width, height) to resize every frame to.
        jpeg_quality: JPEG compression quality 0-100. 90 is a good default.

    Returns:
        A metadata dict with fps, total_frames, duration_s, and output paths.
        This dict is also saved as meta.json inside output_dir.
    """

    # -- checks
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open: {video_path}")

    # -- video properties
    fps           = cap.get(cv2.CAP_PROP_FPS)
    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width_orig    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height_orig   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_s    = total_frames / fps if fps > 0 else 0.0

    print(f"\nVideo : {os.path.basename(video_path)}")
    print(f"  Resolution : {width_orig} x {height_orig}")
    print(f"  FPS        : {fps:.3f}")
    print(f"  Frames     : {total_frames}")
    print(f"  Duration   : {duration_s:.2f}s  ({duration_s/60:.1f} min)")
    print(f"  → Output   : {output_dir}")
    print(f"  → Resize   : {target_size[0]} x {target_size[1]}")
    print()

    # -- extract frames
    encode_params = [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality]
    frame_idx     = 0
    start_time    = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Resize to target size expected by HRNet (width, height)
        frame_resized = cv2.resize(
            frame,
            target_size,
            interpolation=cv2.INTER_LINEAR,
        )

        # Zero-padded filename so files sort correctly (up to 999999 frames)
        out_path = os.path.join(output_dir, f"{frame_idx:06d}.jpg")
        cv2.imwrite(out_path, frame_resized, encode_params)

        frame_idx += 1

        # Progress every 500 frames
        if frame_idx % 500 == 0:
            elapsed   = time.time() - start_time
            pct       = frame_idx / total_frames * 100 if total_frames > 0 else 0
            fps_proc  = frame_idx / elapsed if elapsed > 0 else 0
            remaining = (total_frames - frame_idx) / fps_proc if fps_proc > 0 else 0
            print(
                f"  [{pct:5.1f}%] {frame_idx}/{total_frames} frames"
                f"  |  {fps_proc:.0f} fps  |  ~{remaining:.0f}s remaining"
            )

    cap.release()
    elapsed_total = time.time() - start_time

    print(f"\nDone. Extracted {frame_idx} frames in {elapsed_total:.1f}s")
    print(f"  Avg throughput : {frame_idx / elapsed_total:.0f} frames/s")

    # -- save metadata as JSON
    # IMPORTANT: FPS is saved here so you can convert frame numbers back to
    # timestamps later:  timestamp_s = frame_number / fps
    meta = {
        "video_path":    os.path.abspath(video_path),
        "output_dir":    os.path.abspath(output_dir),
        "fps":           fps,
        "total_frames":  frame_idx,
        "duration_s":    duration_s,
        "target_size":   {"width": target_size[0], "height": target_size[1]},
        "original_size": {"width": width_orig, "height": height_orig},
        "jpeg_quality":  jpeg_quality,
    }

    meta_path = os.path.join(output_dir, "meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"  Metadata saved : {meta_path}")
    print()

    return meta


def frame_to_timestamp(frame_number: int, fps: float) -> float:
    """Convert a frame index to seconds. Use this when matching CSV timestamps."""
    return frame_number / fps


def timestamp_to_frame(timestamp_s: float, fps: float) -> int:
    """Convert a timestamp in seconds to the nearest frame index."""
    return round(timestamp_s * fps)




In [ ]:
video_file = r"C:\Users\elena.graf\Desktop\AA_Deep_Learning\gate-to-gate-tagging\data\swsk_videos\251025_Gut_Soelden_W_GS_run_1.mp4"
out_dir = r"C:\Users\elena.graf\Desktop\AA_Deep_Learning\gate-to-gate-tagging\data\frames"
meta = extract_frames(
    video_path=video_file,
    output_dir=out_dir
)

# Show how to use the saved FPS for timestamp conversion
fps = meta["fps"]
print("Example conversions using saved FPS:")
print(f"  Frame 450  → {frame_to_timestamp(450, fps):.3f}s")
print(f"  4.231s     → frame {timestamp_to_frame(4.231, fps)}")
print()
print("Next step: run parse_annotations.py to match CSV timestamps to frames.")

